# Week 10: Data Preprocessing Pipelines

                **Focus:** Encode categorical data, scale numeric features, and build consistent scikit-learn pipelines.

                **Learning objectives**
                - Compare encoding strategies
- Write a small custom transformer
- Prepare mixed data with a reusable pipeline

                **Source materials:** Data Handling Session new.pptx, Data_Handling_Practical_Lab_.pdf, housing.csv

## Key Highlights

- Ordinal vs one-hot encoding
- Scaling with StandardScaler
- Custom transformers and ColumnTransformer pipelines

## Sample encoding workflow

In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

sample = pd.DataFrame(
    {
        "median_income": [3.2, 5.1, 2.8, 7.3, 4.0],
        "total_rooms": [880, 7099, 1467, 1274, 1627],
        "housing_median_age": [15, 25, 8, 12, 30],
        "ocean_proximity": ["NEAR BAY", "INLAND", "INLAND", "NEAR OCEAN", "ISLAND"],
    }
)

ordinal_encoder = OrdinalEncoder()
sample["ocean_encoded"] = ordinal_encoder.fit_transform(sample[["ocean_proximity"]])
onehot_encoder = OneHotEncoder()
display(sample)
display(onehot_encoder.fit_transform(sample[["ocean_proximity"]]).toarray())


,median_income,total_rooms,housing_median_age,ocean_proximity,ocean_encoded
0,3.2,880,15,NEAR BAY,2.0
1,5.1,7099,25,INLAND,0.0
2,2.8,1467,8,INLAND,0.0
3,7.3,1274,12,NEAR OCEAN,3.0
4,4.0,1627,30,ISLAND,1.0


array([[0., 0., 1., 0.],
       [1., 0., 0., 0.],
       [1., 0., 0., 0.],
       [0., 0., 0., 1.],
       [0., 1., 0., 0.]])

## Custom transformer and full pipeline

In [2]:
from pathlib import Path
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "weeks").exists():
            return candidate
    return current

ROOT = find_project_root()
housing = pd.read_csv(ROOT / "data" / "housing" / "housing.csv")

class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, feature_names):
        self.feature_names = feature_names

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.feature_names)
        X = X.copy()
        X["rooms_per_household"] = X["total_rooms"] / X["households"]
        X["population_per_household"] = X["population"] / X["households"]
        X["bedrooms_per_room"] = X["total_bedrooms"] / X["total_rooms"]
        return X

features = housing.drop(columns=["median_house_value"])
num_cols = features.drop(columns=["ocean_proximity"]).columns.tolist()
cat_cols = ["ocean_proximity"]

num_pipeline = Pipeline([
    ("adder", CombinedAttributesAdder(num_cols)),
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

prepared = full_pipeline.fit_transform(features)
print(type(prepared))
print(prepared.shape)


<class 'numpy.ndarray'>
(20640, 16)


## Practice Tasks

- Compare encodings: Explain why one-hot encoding is safer for unordered categories.
- Build a pipeline: Create a numeric and categorical preprocessing workflow.